# Opbygning af en telekom-omsætningssikringsterning med PROC SUMMARY

## Resumé

Et omsætningssikringsteam hos et mobilselskab foraggregerer en måneds abonnent-dag-faktureringsdata i en kompakt opsummeringsterning, så analytikere kan drille ned i afregnet omsætning fordelt på region, abonnementsniveau og opkaldstype uden at skulle genscanne den rå faktatabel. `PROC SUMMARY` ruller 100 abonnent-dag-poster op i et multidimensionelt sæt af celler: det mest detaljerede niveau (region x abonnementsniveau x opkaldstype) fylder 25 af 27 mulige celler, mens navngivne delterninger giver de marginaler, analytikerne oftest forespørger. I denne eksempelmåned afregnede selskabet \$222.78 på tværs af de tre aktive regioner, hvor Syd (\$97.44) og Øst (\$86.94) tilsammen udgjorde 83% af omsætningen, mens Nord lå bagefter med \$38.40. Den rigeste enkelte delterning er Plus-niveau Tale-tjenesten (\$59.06 over 18 poster), og en rangering af celler efter omsætning pr. post afslører data-forbrugsceller som de mest værdifulde mål for en lækagerevision (højeste udbytte \$7.87/post). Alle tal nedenfor er aflæst direkte fra den udførte output.

## Datakilder

| Datasæt | Rækker | Niveau | Nøglevariabler |
|---------|------|-------|---------------|
| `work.cdr_billing` | 100 | Én række pr. abonnent-dag-forbrugsopsummering | `region` (Øst/Syd/Nord), `plan_tier` (Forudbetalt/Basis/Plus), `call_type` (Tale/SMS/Data), `device_class`, `bill_month`, `revenue`, `call_minutes`, `data_mb`, `subscriber_wt` |
| `work.cube_nway` | 25 | Én række pr. ikke-tom (region x plan_tier x call_type)-celle | `_FREQ_`, `rev_total`, `rev_mean`, `rev_max`, `min_total`, `data_total` |
| `work.cube_hier` | 22 | Én række pr. celle i de navngivne drill-delterninger | `_TYPE_`, `_FREQ_`, `rev_total` |

Alle data genereres inline med `call streaminit()` / `rand()`; ingen eksterne filer eller netværksadgang er nødvendig. Dette miljø kører ulicenseret, så skrevne datasæt er begrænset til 100 observationer - scenariet er dimensioneret, så terningen er fuldt udfyldt inden for denne grænse.

## Fra rå opkaldsdetaljeposter til en terning, man kan drille i

Mobilselskaber afregner omsætning på tværs af millioner af daglige forbrugshændelser. For at lade omsætningssikringsanalytikere besvare spørgsmål som *"Hvad var Plus-niveau tale-omsætningen i Syd-regionen sidste måned?"* uden at genscanne den rå faktatabel hver gang, **foraggregerer** vi data i en kompakt opsummeringsterning.

`PROC SUMMARY` er Base SAS's arbejdshest til præcis dette: den grupperer en flad faktatabel efter en eller flere `CLASS`-dimensioner og skriver de ønskede statistikker til et output-datasæt, hvor hver række mærkes med `_TYPE_` (hvilke dimensioner er aktive) og `_FREQ_` (antal poster bag cellen). Dette output-datasæt *er* en multidimensionel terning - den samme oprulningsstruktur, som et OLAP-værktøj ville eksponere, materialiseret som et almindeligt SAS-datasæt, du kan udskrive, joine og udsnitte.

Denne notesbog:

1. Genererer en realistisk måneds abonnent-dag-faktureringsposter.
2. Opbygger den mest detaljerede terning (region x abonnementsniveau x opkaldstype) med `PROC SUMMARY NWAY`.
3. Materialiserer navngivne **drill-delterninger** med `TYPES`-sætningen.
4. Projicerer omsætning til abonnentgrundlaget med en `WEIGHT`, drilller ned i én region og rangerer celler efter omsætning pr. post til lækagetriage.

## Trin 1 - Generer syntetiske opkaldsdetalje-/faktureringsdata

Hver række opsummerer én abonnents forbrug af én tjeneste på én dag. Vi bruger `call streaminit` for reproducerbarhed og `rand()` til at trække plausible fordelinger: omsætning og forbrug skalerer med abonnementsniveauet, taleomsætning følger fakturerbare minutter, og dataomsætning følger megabyte. Hver `RAND('table', ...)` får én sandsynlighed pr. kategori, så alle regioner, niveauer og opkaldstyper optræder i stikprøven på 100 poster. En lille `subscriber_wt`-undersøgelsesvægt tilføjes, så vi senere kan demonstrere et vægtet mål.

In [1]:
data work.cdr_billing;
    CALL streaminit(20260131);
    LÆNGDE region $6 plan_tier $12 call_type $7 device_class $8 bill_month $7;
    MÆRKAT revenue       = "Afregnet omsætning (USD)"
          call_minutes  = "Fakturerbare taleminutter"
          data_mb       = "Datamængde (MB)"
          subscriber_wt = "Abonnentundersøgelsesvægt"
          region        = "Region"
          plan_tier     = "Abonnementsniveau"
          call_type     = "Opkaldstype"
          device_class  = "Enhedstype"
          bill_month    = "Faktureringsmåned";

    GØR i = 1 TIL 100;
        /* --- Dimensions: one probability per level, summing to 1.0 --- */
        r = rand("table", 0.40, 0.33, 0.27);
        VÆLG (r);
            NÅR (1) region = "Øst";
            NÅR (2) region = "Syd";
            ELLERS_OM region = "Nord";
        SLUT;

        p = rand("table", 0.30, 0.40, 0.30);
        VÆLG (p);
            NÅR (1) plan_tier = "Forudbetalt";
            NÅR (2) plan_tier = "Basis";
            ELLERS_OM plan_tier = "Plus";
        SLUT;

        c = rand("table", 0.50, 0.30, 0.20);
        VÆLG (c);
            NÅR (1) call_type = "Tale";
            NÅR (2) call_type = "SMS";
            ELLERS_OM call_type = "Data";
        SLUT;

        HVIS rand("uniform") < 0.55 SÅ device_class = "Smart";
        ELLERS device_class = "Basal";

        bill_month = "2026-01";

        /* --- Measures, scaled by tier and service --- */
        tier_mult = (plan_tier = "Forudbetalt")*0.7
                  + (plan_tier = "Basis")*1.0
                  + (plan_tier = "Plus")*1.7;

        call_minutes = round((call_type = "Tale")
                       * rand("gamma", 2.0) * 18 * tier_mult, 0.1);
        data_mb      = round((call_type = "Data")
                       * rand("gamma", 1.5) * 220 * tier_mult, 0.1);

        base_rev = 0.05*call_minutes + 0.010*data_mb
                 + (call_type = "SMS") * rand("poisson", 30) * 0.03;
        revenue = round(base_rev * (0.85 + 0.30*rand("uniform")), 0.01);

        subscriber_wt = round(0.8 + rand("uniform")*1.6, 0.01);

        UDDATA;
    SLUT;
    FJERN i r p c tier_mult base_rev;
KØR;

PROC PRINT data=work.cdr_billing(obs=8) MÆRKAT noobs;
    TITEL "Eksempel på abonnent-dag-faktureringsposter";
KØR;

                                      Eksempel på abonnent-dag-faktureringsposter                                       

Region  Abonnementsniveau  Opkaldstype  Enhedstype   Faktureringsmåned  Fakturerbare taleminutter   Datamængde (MB)   Afregnet omsætning (USD)    Abonnentundersøgelsesvægt
Nord    Plus               SMS          Smart       2026-01                                     0                 0                       0.67                         1.13
Syd     Forudbetalt        SMS          Basal       2026-01                                     0                 0                       0.94                         1.42
Syd     Plus               SMS          Smart       2026-01                                     0                 0                       0.99                         0.86
Syd     Plus               SMS          Smart       2026-01                                     0                 0                       1.01                         1.03
Syd     Plus      


NOTE: DATA work.cdr_billing


NOTE: Wrote work.cdr_billing (100 rows, 9 columns).
NOTE: DATA elapsed:
  wall  0.04 seconds
  cpu   0.04 seconds
NOTE: PROC PRINT data=work.cdr_billing

NOTE: PROC PRINT completed: 8 observations printed, 9 variables


## Trin 2 - Opbyg den mest detaljerede terning med PROC SUMMARY NWAY

`NWAY` beholder kun den enkelte mest detaljerede kombination af alle `CLASS`-dimensioner - her hver udfyldt (region x plan_tier x call_type)-celle. For hver celle gemmer vi omsætnings-`SUM`, `MEAN` og `MAX` plus de samlede minutter og megabyte, så en analytiker kan aflæse total omsætning pr. celle, udlede gennemsnittet pr. post og finde den største enkeltafregning. `_FREQ_` angiver, hvor mange abonnent-dage der ligger bag hver celle. Vi dropper `_TYPE_` her, fordi hver række har samme type på NWAY-niveauet.

In [2]:
PROC SUMMARY data=work.cdr_billing NWAY;
    KLASSE region plan_tier call_type;
    VARIABEL revenue call_minutes data_mb;
    UDDATA out=work.cube_nway(FJERN=_type_)
           sum(revenue)=rev_total  mean(revenue)=rev_mean  MAX(revenue)=rev_max
           sum(call_minutes)=min_total
           sum(data_mb)=data_total;
KØR;

PROC PRINT data=work.cube_nway(obs=12) noobs MÆRKAT;
    TITEL "NWAY-terningsceller (region * plan_tier * call_type)";
    MÆRKAT region="Region" plan_tier="Abonnementsniveau" call_type="Opkaldstype"
          _freq_="Antal poster" rev_total="Omsætning i alt" rev_mean="Gns. omsætning"
          rev_max="Maks. omsætning" min_total="Minutter i alt" data_total="Data i alt (MB)";
    format rev_total rev_mean rev_max min_total data_total comma10.2;
KØR;

                                  NWAY-terningsceller (region * plan_tier * call_type)                                  

Region  Abonnementsniveau  Opkaldstype  Antal poster   Omsætning i alt   Gns. omsætning   Maks. omsætning  Minutter i alt  Data i alt (MB)
Nord    Basis              Data                    1              7.87             7.87              7.87            0.00           725.10
Nord    Basis              SMS                     2              1.95             0.97              1.17            0.00             0.00
Nord    Basis              Tale                    4              4.74             1.19              2.35           91.00             0.00
Nord    Forudbetalt        Data                    2              2.16             1.08              1.09            0.00           229.50
Nord    Forudbetalt        SMS                     2              2.00             1.00              1.18            0.00             0.00
Nord    Forudbetalt        Tale             


NOTE: PROC MEANS
NOTE: Output dataset work.cube_nway has 25 observations and 9 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 12 observations printed, 9 variables


## Trin 3 - Materialiser navngivne drill-delterninger med TYPES

En OLAP-terning forudlagrer de oprulninger, analytikere navigerer mest. `TYPES`-sætningen gør præcis det: hvert udtryk beder `PROC SUMMARY` om at udsende én delterning. Vi anmoder om hovedtotalen `()`, region-marginalen samt de to-vejs-delterninger `region*plan_tier` og `call_type*plan_tier` - de drill-stier, et omsætningsdashboard ville eksponere. SAS mærker hver delterning med en `_TYPE_`-kode (en bitmaske over `CLASS`-listen), så et enkelt output-datasæt bærer hvert niveau af hierarkiet.

In [3]:
PROC SUMMARY data=work.cdr_billing;
    KLASSE region plan_tier call_type;
    VARIABEL revenue;
    TYPES () region region*plan_tier call_type*plan_tier;
    UDDATA out=work.cube_hier
           sum(revenue)=rev_total;
KØR;

PROC PRINT data=work.cube_hier noobs MÆRKAT;
    TITEL "Delterning-oprulninger: hovedtotal, region, region*niveau, opkaldstype*niveau";
    VARIABEL _type_ region plan_tier call_type _freq_ rev_total;
    MÆRKAT _type_="Type" region="Region" plan_tier="Abonnementsniveau" call_type="Opkaldstype"
          _freq_="Antal poster" rev_total="Omsætning i alt";
    format rev_total comma10.2;
KØR;

                     Delterning-oprulninger: hovedtotal, region, region*niveau, opkaldstype*niveau                      

Type  Region  Abonnementsniveau  Opkaldstype  Antal poster   Omsætning i alt
   0                                                   100            222.78
   3          Basis              Data                    8             38.06
   3          Basis              SMS                     8              8.03
   3          Basis              Tale                   20             42.33
   3          Forudbetalt        Data                    7             14.50
   3          Forudbetalt        SMS                     7              6.82
   3          Forudbetalt        Tale                   16             24.77
   3          Plus               Data                    3             17.46
   3          Plus               SMS                    13             11.75
   3          Plus               Tale                   18             59.06
   4  Nord                     


NOTE: PROC MEANS
NOTE: Output dataset work.cube_hier has 22 observations and 6 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_hier

NOTE: PROC PRINT completed: 22 observations printed, 6 variables


## Trin 4 - Vægtet projektion, en regional drill og lækagetriage

Tre opslag et omsætningssikringsteam faktisk udfører mod terningen:

- **Vægtet projektion.** Ved at tilføje `WEIGHT=subscriber_wt` til en `region*plan_tier`-opsummering rapporteres omsætning skaleret til hele det abonnentgrundlag, stikprøven repræsenterer, i stedet for de rå udtagne rækker.
- **Drill.** Ved at filtrere NWAY-terningen til én region udvides én gren af hierarkiet - her Syd - til dens niveau-for-tjeneste-detalje.
- **Lækagetriage.** Ved at sortere cellerne efter gennemsnitlig omsætning pr. post fremhæves cellerne med højest udbytte; celler med lav frekvens og højt udbytte er præcis dem, en revision undersøger for fejlprissatte eller lækkende indtægter.

In [4]:
/* Weighted revenue projected to the subscriber base */
PROC SUMMARY data=work.cdr_billing NWAY;
    KLASSE region plan_tier;
    VARIABEL revenue;
    VÆGT subscriber_wt;
    UDDATA out=work.cube_wt(FJERN=_type_ _freq_)
           sum(revenue)=rev_weighted  n(revenue)=records;
KØR;

PROC PRINT data=work.cube_wt noobs MÆRKAT;
    TITEL "Vægtet omsætning efter region * abonnementsniveau (projiceret til abonnentgrundlag)";
    MÆRKAT region="Region" plan_tier="Abonnementsniveau" rev_weighted="Vægtet omsætning" records="Antal poster";
    format rev_weighted comma10.2;
KØR;

/* Drill into the South region branch of the cube */
PROC PRINT data=work.cube_nway noobs MÆRKAT;
    HVOR region = "Syd";
    TITEL "Drill ned i Syd: omsætningsceller efter niveau og opkaldstype";
    VARIABEL plan_tier call_type _freq_ rev_total rev_mean;
    MÆRKAT plan_tier="Abonnementsniveau" call_type="Opkaldstype" _freq_="Antal poster"
          rev_total="Omsætning i alt" rev_mean="Gns. omsætning";
    format rev_total rev_mean comma10.2;
KØR;

/* Rank cells by per-record yield for leakage triage */
PROC SORT data=work.cube_nway out=work.cube_ranked;
    EFTER FALDENDE rev_mean;
KØR;

PROC PRINT data=work.cube_ranked(obs=6) noobs MÆRKAT;
    TITEL "Celler med højest gennemsnitlig omsætning (udbytte pr. post)";
    VARIABEL region plan_tier call_type _freq_ rev_mean rev_max;
    MÆRKAT region="Region" plan_tier="Abonnementsniveau" call_type="Opkaldstype"
          _freq_="Antal poster" rev_mean="Gns. omsætning" rev_max="Maks. omsætning";
    format rev_mean rev_max comma10.2;
KØR;

                  Vægtet omsætning efter region * abonnementsniveau (projiceret til abonnentgrundlag)                   

Region  Abonnementsniveau    Vægtet omsætning  Antal poster
Nord    Basis                           18.30             7
Nord    Forudbetalt                     20.56             9
Nord    Plus                            22.42             7
Syd     Basis                           58.63            14
Syd     Forudbetalt                     27.77            10
Syd     Plus                            56.29            15
Øst     Basis                           50.85            15
Øst     Forudbetalt                     29.77            11
Øst     Plus                            59.59            12

                             Drill ned i Syd: omsætningsceller efter niveau og opkaldstype                              

Abonnementsniveau  Opkaldstype  Antal poster   Omsætning i alt   Gns. omsætning
Basis              Data                    3             12.02             


NOTE: PROC MEANS
NOTE: Output dataset work.cube_wt has 9 observations and 4 variables.
NOTE: PROC MEANS statement used.
NOTE: PROC PRINT data=work.cube_wt

NOTE: PROC PRINT completed: 9 observations printed, 4 variables
NOTE: PROC PRINT data=work.cube_nway

NOTE: PROC PRINT completed: 9 observations printed, 5 variables
NOTE: PROC SORT data=work.cube_nway

NOTE: Unlicensed mode - input limited to 100 observations.
NOTE: Read 25 rows from work.cube_nway.
NOTE: Wrote work.cube_ranked (25 rows, 9 columns).
NOTE: PROC SORT statement used.
NOTE: PROC PRINT data=work.cube_ranked

NOTE: PROC PRINT completed: 6 observations printed, 6 variables


## Fortolkning af resultaterne

Opsummeringsterningen omdanner 100 rå abonnent-dag-rækker til et kompakt sæt af foraggregerede celler, der besvarer drill-down-spørgsmål øjeblikkeligt uden at genscanne faktatabellen:

- **Hvor omsætningen ligger.** Region-marginalen (`_TYPE_=4`) viser, at Syd afregnede \$97.44 og Øst \$86.94 - tilsammen 83% af månedens \$222.78 - mens Nord bidrog med \$38.40. Inden for `call_type*plan_tier`-delterningen (`_TYPE_=3`) er Plus-niveau Tale den enkelte rigeste celle med \$59.06 over 18 poster, med Basis-niveau Tale som nummer to med \$42.33.
- **Vægtet projektion.** Anvendelse af undersøgelsesvægten omfordeler rangeringen mod abonnementer, hvis abonnenter vejer tungere: Øst-Plus (\$59.59) og Syd-Basis (\$58.63) fører den projicerede `region*plan_tier`-omsætning - et andet billede end de uvægtede regionstotaler og en påmindelse om at rapportere projiceret frem for udtaget omsætning ved dimensionering af eksponering.
- **Omsætning pr. post og lækagetriage.** Rangering af NWAY-celler efter `rev_mean` placerer data-forbrugsceller øverst - Nord-Basis-Data (\$7.87/post) og Syd-Plus-Data (\$5.96/post) - hvilket bekræfter, at datatungt forbrug driver den højeste omsætning pr. post. Cellerne med lav frekvens (én eller to poster) er præcis dem, en omsætningssikringsanalytiker ville hente de underliggende opkaldsdetaljeposter for, for at bekræfte, at den høje afregning er korrekt prissat og ikke en fejl.

For et omsætningssikringsteam er denne terning grundlaget for variansdashboards: sammenlign afregnet omsætning med forventet listepris-omsætning pr. (region, niveau, opkaldstype)-celle, og de celler, hvis gennemsnit eller vægtede total afviger mest, bliver de lækagesager, det er værd at revidere. Fordi hele strukturen er et almindeligt SAS-datasæt, kan næste måneds terning unioneres, sammenlignes eller kobles til en listepris med de samme Base SAS-værktøjer.